In [1]:
from data.opendataloader import OpenDataLoader
from fastml.modules.cnn import load_qCNN
from fastml.utils.image import pad
from fastml.utils_egamma.efex import eFex_slidingwindow_mask
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc


In [2]:
model_path = 'models/cls_qcnn/baseline_model.keras'

In [32]:
signal = ak.from_parquet('/workspace/workDir/test_data/test_signal.parquet')
background = ak.from_parquet('/workspace/workDir/test_data/test_background.parquet')

In [33]:
model = load_qCNN(model_path)

In [34]:
predictions = []
for dataset in [signal, background]:
    pred = model.predict(
        np.array(ak.flatten(dataset.image)),
        batch_size=1024,
        verbose=0
    )
    
    pred = pred[:,0,0,0]
    counts = ak.num(dataset.seed_info)
    predictions.append(ak.unflatten(pred, counts))
    
predictions = {
    "signal" : predictions[0],
    "background" : predictions[1]
}

In [35]:
y_sig = ak.flatten(predictions['signal'])
y_bkg = ak.flatten(predictions['background'][:len(y_sig)])

y = np.concatenate([np.ones_like(y_sig, dtype=int), np.zeros_like(y_bkg, dtype=int)])
s = np.concatenate([y_sig, y_bkg])

fpr, tpr, _ = roc_curve(y, s)
A = auc(fpr, tpr)

fake_rej = 1 - fpr

plt.figure(figsize=(6, 6), dpi=200)
plt.plot(tpr, fake_rej, ".", ms=3, color=plt.cm.cool(0.5))

plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.xlabel("Signal Efficiency (TPR)")
plt.ylabel("Fake Rejection")
plt.grid(True, linestyle="--", alpha=0.5)
plt.title(f"ROC Curve, AUC = {A:.3f}")
plt.show()

In [36]:
bin_edges = np.arange(0, 1, 0.05)

plt.figure(figsize=(10, 4), dpi=200)

hist, _ = np.histogram(ak.flatten(predictions['signal']), bins=bin_edges, density = True)
plt.stairs(hist, bin_edges, label=f"sig", color=f"blue")

hist2, _ = np.histogram(ak.flatten(predictions['background']), bins=bin_edges, density = True)
plt.stairs(hist2, bin_edges, label=f"bkg", color=f"red")

plt.xlabel("score")
plt.ylabel('AU')
plt.xlim(0,1)
plt.legend()
plt.show()